# MySketch — AI Sketch Completion (Gradio + SDXL + ControlNet)

Рисуй набросок → AI дорисовывает.

**Как использовать:**
1. Runtime → Run all (или Shift+Enter на каждой ячейке)
2. Дождаться загрузки модели (~2-3 мин на T4)
3. Появится публичная ссылка вида `https://xxxx.gradio.live`
4. Открыть её в браузере — рисовать на холсте и жать «Generate»
5. Или использовать ссылку как `FOOOCUS_URL` для вашего прокси-сервера

## ⚙️ Шаг 1: Проверка GPU

In [ ]:
import torch
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 📦 Шаг 2: Установка зависимостей

In [ ]:
!pip install -q diffusers transformers accelerate controlnet_aux safetensors gradio pillow

## 🎨 Шаг 3: Загрузка SDXL + ControlNet

In [ ]:
import torch
from diffusers import StableDiffusionXLControlNetPipeline, ControlNetModel, AutoencoderKL
from PIL import Image
import numpy as np

print("Loading ControlNet (scribble)...")
controlnet = ControlNetModel.from_pretrained(
    "diffusers/controlnet-scribble-sdxl-1.0",
    torch_dtype=torch.float16,
).to("cuda")

print("Loading VAE (fp16 fix)...")
vae = AutoencoderKL.from_pretrained(
    "madebyollin/sdxl-vae-fp16-fix",
    torch_dtype=torch.float16
).to("cuda")

print("Loading SDXL pipeline...")
pipe = StableDiffusionXLControlNetPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    controlnet=controlnet,
    vae=vae,
    torch_dtype=torch.float16,
    variant="fp16",
    use_safetensors=True,
).to("cuda")

# Экономия памяти для T4 (15 GB)
pipe.enable_model_cpu_offload()
pipe.enable_vae_slicing()
pipe.enable_vae_tiling()

print("✅ Model loaded successfully!")

## 🚀 Шаг 4: Запуск Gradio-интерфейса

In [ ]:
import gradio as gr
import io
import base64
import requests

DEFAULT_PROMPT = (
    "finish this sketch, make it a complete cute illustration, "
    "add creative details, harmonious colors, professional digital art"
)

def generate(sketch, prompt):
    """Принимает скетч (numpy array), возвращает готовое изображение"""
    if sketch is None:
        return None
    
    # Gradio даёт RGBA, конвертируем в RGB
    img = Image.fromarray(sketch.astype('uint8'), 'RGBA').convert('RGB')
    img = img.resize((1024, 1024), Image.LANCZOS)
    
    # Генерация
    result = pipe(
        prompt=prompt if prompt else DEFAULT_PROMPT,
        negative_prompt="ugly, deformed, blurry, low quality, bad anatomy",
        image=img,
        controlnet_conditioning_scale=0.8,
        guidance_scale=7.5,
        num_inference_steps=30,
    ).images[0]
    
    return result


# Gradio-интерфейс: холст + кнопка + результат
with gr.Blocks(title="MySketch", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🎨 MySketch")
    gr.Markdown("Нарисуй что-нибудь на холсте и нажми **Generate**")
    
    with gr.Row():
        with gr.Column():
            sketch = gr.ImageEditor(
                label="Твой скетч",
                type="numpy",
                width=512,
                height=512,
                brush=gr.Brush(default_size=5),
            )
            prompt_input = gr.Textbox(
                label="Prompt (описание)",
                placeholder=DEFAULT_PROMPT,
            )
            btn = gr.Button("✨ Generate", variant="primary", size="lg")
        with gr.Column():
            output = gr.Image(label="Результат")
    
    btn.click(fn=generate, inputs=[sketch, prompt_input], outputs=output)


# Запуск с публичной ссылкой (share=True — как в Fooocus)
print("\n" + "="*60)
print("  🔗 Загружаю интерфейс... публичная ссылка будет ниже")
print("="*60 + "\n")

demo.launch(share=True, debug=False)

# Для прокси-сервера: эндпоинт /api/predict
# URL будет вида: https://XXXXX.gradio.live/api/predict
# Подключение: set FOOOCUS_URL=https://XXXXX.gradio.live